# Attention-model classification accuracy (Figure 3B)

Top-1 classification accuracy of the Set-Transformer as a function of the number of cells per bag, for the **phase** (label-free) classifier and the **all-fluorescence** (combined all-channel) classifier. Two panels: gene knockouts (C = 1,001) and protein complexes (C = 99). Each series is the **mean Top-1 accuracy** per `n_cells` bin (every row is balanced at 50 bags, so the mean equals the overall accuracy = P(correct on a random bag of that size)); the shaded band is **mean &plusmn; 1 SD**. The x-axis is log-scaled.

Inputs are the per-class evaluation CSVs, curated into `../../../data/figures/figure_3/`. Protein-complex labels come from the EBI complex annotations.

| file | level | modality |
| --- | --- | --- |
| `figure_3b_gene_level_phase.csv` | gene KO | phase |
| `figure_3b_gene_level_fluorescence.csv` | gene KO | all fluorescence |
| `figure_3b_ebi_level_phase.csv` | protein complex (EBI) | phase |
| `figure_3b_ebi_level_fluorescence.csv` | protein complex (EBI) | all fluorescence |

> **Note on the complex panel.** The complex CSVs have one row per *gene* (311 genes) carrying its
> `label_name` (complex, 99 unique), so grouping by `n_cells` alone averages over genes, not over
> complexes — complexes with more member genes (up to 15) are weighted more heavily. This reads
> 0.4–7.2 percentage points higher than a true per-complex mean. See the summary-table cell.


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import LogLocator, MultipleLocator, NullFormatter

# Keep text editable in Illustrator (SVG keeps <text> elements; PDF uses TrueType).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

FIGURES_DIR = Path("../../../output/figure_3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data paths

Per-class evaluation CSVs, read from the central figure-data directory
`../../../data/figures/figure_3/`.

In [ ]:
FIGURE_DATA = Path("../../../data/figures/figure_3")

# (panel title, phase csv, all-fluorescence csv)
# "gene_level" = gene KO / NTC (1,001 classes); "ebi_level" = EBI protein
# complexes (99 classes); "fluorescence" = the combined all-channel classifier.
PANELS = [
    ("Gene KO\nclassification",
     FIGURE_DATA / "figure_3b_gene_level_phase.csv",
     FIGURE_DATA / "figure_3b_gene_level_fluorescence.csv"),
    ("Protein complex\nclassification",
     FIGURE_DATA / "figure_3b_ebi_level_phase.csv",
     FIGURE_DATA / "figure_3b_ebi_level_fluorescence.csv"),
]

## Configuration

In [ ]:
ACC_COL = "top1_acc"        # "top1_acc" or "top5_acc"
PHASE_COLOR = "#3a3a3a"     # phase = dark gray/black
FLUOR_COLOR = "#d10a7d"     # all fluorescence = magenta

## Helpers

`mean_std` returns the mean Top-1 (%) across classes per `n_cells` bin and the &plusmn;1 SD band (clipped to [0, 100%]).

In [ ]:
def mean_std(df):
    g = df.groupby("n_cells")[ACC_COL]
    m, sd = g.mean() * 100, g.std() * 100
    lo = (m - sd).clip(lower=0)
    hi = (m + sd).clip(upper=100)
    return m.index.values, m.values, lo.values, hi.values

## Figure 3B

Two panels sharing the y-axis; log x-axis with explicit cell-count ticks. Saves an SVG (paper) + PNG.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8), sharey=True)

for ax, (title, phase_csv, fluor_csv) in zip(axes, PANELS):
    for csv, color in [(phase_csv, PHASE_COLOR), (fluor_csv, FLUOR_COLOR)]:
        df = pd.read_csv(csv, usecols=["n_cells", ACC_COL])
        x, m, lo, hi = mean_std(df)
        ax.fill_between(x, lo, hi, color=color, alpha=0.09, linewidth=0, zorder=2)
        ax.plot(x, m, "-o", color=color, linewidth=4, markersize=9,
                markerfacecolor="black", markeredgecolor="black", markeredgewidth=0.8, zorder=3)

    ax.set_xscale("log")
    ax.set_xticks([10, 100, 1000, 5000])
    ax.set_xticklabels(["10", "100", "1000", "5000"])
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_ylim(0, 105)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_yticklabels([f"{v}%" for v in [0, 20, 40, 60, 80, 100]])
    ax.yaxis.set_minor_locator(MultipleLocator(10))
    ax.set_title(title, fontsize=42, pad=14)
    ax.set_xlabel("# cells per bag", fontsize=42)
    ax.tick_params(axis="both", which="major", labelsize=36, width=2.5, length=13)
    ax.tick_params(axis="both", which="minor", width=2, length=7)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_linewidth(2)
    ax.legend([Patch(facecolor=PHASE_COLOR), Patch(facecolor=FLUOR_COLOR)],
              ["Phase", "All fluo."], loc="lower right", fontsize=28,
              frameon=False, handlelength=1.4, handleheight=1.1)

axes[0].set_ylabel("Classification accuracy", fontsize=42)
fig.tight_layout(w_pad=4)
fig.savefig(FIGURES_DIR / "eval_accuracy_curves.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "eval_accuracy_curves.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Mean &plusmn; SD Top-1 accuracy per (level, modality, n_cells).

In [ ]:
rows = []
for (title, phase_csv, fluor_csv) in PANELS:
    level = title.split("\n")[0]
    for csv, modality in [(phase_csv, "phase"), (fluor_csv, "all_fluor")]:
        df = pd.read_csv(csv)
        # Gene-level CSVs key on gene_name; complex-level CSVs add label_name
        # (the complex) on top of the per-gene rows.
        id_col = "label_name" if "label_name" in df.columns else "gene_name"
        g = df.groupby("n_cells")[ACC_COL]
        for n in sorted(g.groups):
            vals = g.get_group(n).to_numpy()
            sub = df[df["n_cells"] == n]
            # n_rows is what acc_mean/acc_std actually average over; n_classes is
            # the number of distinct labels. They differ for the complex panel
            # (311 gene rows vs 99 complexes), so acc_mean there is gene-weighted.
            # acc_mean_per_class re-averages within each label first.
            rows.append({"level": level, "modality": modality, "n_cells": int(n),
                         "n_rows": int(vals.size),
                         "n_classes": int(sub[id_col].nunique()),
                         "acc_mean": float(np.mean(vals)),
                         "acc_std": float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0,
                         "acc_mean_per_class": float(sub.groupby(id_col)[ACC_COL].mean().mean())})

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "eval_accuracy_curves_summary.csv", index=False)
summary